# ARC-AGI-3 DuckWADL — Integrated AI Difference Learning + DifferenceFusion

**Core loop:** `observe → build current-game difference state → compare EXPLOIT vs EXPLORE → fuse evidence → act once → measure actual delta → update ADL → next move`.

This build integrates ADL directly into the Duck action loop. It preserves the strict single-environment/no-cross-game-prior contract while adding explicit difference signatures, weighted DifferenceFusion, prediction calibration, current-game difference memory, and post-run coverage/quality auditing.


In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Bound generation before inference modules are imported. The original run
# allowed unlimited tool steps/output, amplifying stalls under high fan-out.
_score_runtime_env = {
    'LOCAL_ANALYZER_MAX_OUTPUT': os.environ.get('TAAF_MAX_OUTPUT_TOKENS', '8192'),
    'LOCAL_ANALYZER_TOOL_STEPS': os.environ.get('TAAF_TOOL_STEPS', '8'),
    'LOCAL_ANALYZER_TEMPERATURE': os.environ.get('TAAF_TEMPERATURE', '0.6'),
    'LOCAL_ANALYZER_TOP_P': os.environ.get('TAAF_TOP_P', '0.95'),
}
os.environ.update(_score_runtime_env)
_persisted_setup_env = json.loads(SETUP_ENV_PATH.read_text())
_persisted_setup_env.update(_score_runtime_env)
SETUP_ENV_PATH.write_text(json.dumps(_persisted_setup_env, indent=2, sort_keys=True) + '\n')
print(f'taaf.kaggle: bounded analyzer controls = {_score_runtime_env}')


## 4.1 Minimal Duck Harness compatibility

Keep the bundled solver unchanged except for the missing neutral `ACTION7` reverse mapping. No prompt, score, environment, or execution method is patched.


In [ ]:
import inference.agent.action_names as action_names
import inference.framework.solver as solver_module

action_names.MODEL_TO_ENGINE_ACTION["ACTION7"] = "ACTION7"
assert action_names.to_model_action("ACTION7") == "ACTION7"
assert action_names.to_engine_action("ACTION7") == "ACTION7"

PATCH_STATUS = {
    "patch": "minimal-action7-reverse-map-v1",
    "action7_reverse_mapping": True,
    "system_prompt_changed": False,
    "solver_methods_changed": False,
    "dataset_modified": False,
}
print(f"taaf.kaggle: compatibility={PATCH_STATUS}")


## 4.2 Hard no-prior runtime contract

Declare and verify the information boundary before loading the benchmark. Only current-game observations, actions, rewards, transitions, Qwen's pass-1 transcript, and Gemma's review of that transcript may cross into pass 2.


In [ ]:
NO_PRIOR_CONTRACT = {
    "allowed": [
        "same_current_run_same_game_observations",
        "same_current_run_same_game_actions",
        "same_current_run_same_game_rewards",
        "same_current_run_same_game_transitions",
        "same_current_run_same_game_post_move_adl",
        "same_current_run_same_game_dual_path_decisions",
    ],
    "forbidden": [
        "historical_transcripts",
        "yesterday_transcripts",
        "routebooks",
        "replays",
        "solved_paths",
        "hidden_labels",
        "cross_run_state",
        "cross_game_state",
        "prior_submission_state",
    ],
}
assert set(NO_PRIOR_CONTRACT["allowed"]).isdisjoint(
    NO_PRIOR_CONTRACT["forbidden"]
)
print("taaf.kaggle: strict current-game/current-run no-prior contract active")


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6. Fixed real-run configuration

Run every discovered game with concurrency 4 and strict no-prior behavior. A dynamic per-game cap keeps both complete passes inside Kaggle's nine-hour GPU runtime limit. No environment can be omitted.


In [ ]:
STRICT_NO_PRIOR = True
TARGET_CONCURRENCY = 4

_original_game_budget = float(
    getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0
)
_original_concurrency = max(
    1, int(getattr(bm.solver, "concurrency", 1) or 1)
)
bm.solver.concurrency = TARGET_CONCURRENCY

# Optional grafts are constrained to the same current game and current run.
# Banking and transfer are intentionally never installed.
try:
    from taaf_grafts.composite import install as _install_taaf_grafts
except ModuleNotFoundError:
    _install_taaf_grafts = None

_graft_flags = {
    "shortcircuit": True,
    "efficiency": True,
    "retry_guard": True,
    "recovery": True,
    "context_window": int(
        os.environ.get("TAAF_CONTEXT_WINDOW", "32768")
    ),
}
if _install_taaf_grafts is not None:
    _install_taaf_grafts(bm, _graft_flags, expected_version=1)

assert bm.solver.concurrency == TARGET_CONCURRENCY
assert "banking" not in _graft_flags
assert "transfer" not in _graft_flags
print(
    "REAL RUN CONFIG: "
    f"strict_no_prior={STRICT_NO_PRIOR} "
    f"concurrency={TARGET_CONCURRENCY} "
    "games_required=all_discovered "
    f"source_per_game_budget={_original_game_budget}"
)


## 7. Integrated DuckWADL ADL + DifferenceFusion policy

This cell upgrades the Duck analyzer from prompt-only dual-path selection into an explicit ADL architecture: a current-game Difference Memory schema, semantic difference signatures, weighted DifferenceFusion, prediction calibration, and mandatory post-move learning after every committed action. Candidate A/B comparison remains internal; only the selected action touches the real environment.


In [ ]:
# === DUCKWADL INTEGRATED ADL + DIFFERENCEFUSION — BEFORE AND AFTER EVERY MOVE ===
import json
import os
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional
from inference.agent.tool_agent import ToolAgent

DUCKWADL_ADL_ENABLED = True
POST_MOVE_ADL_ENABLED = True
DUAL_PATH_CANDIDATES = 2
ADL_SCHEMA = "adl.arc3.duckwadl.differencefusion.v3"

DUAL_PATH_POLICY_LOG = WORKING_DIR / "dual_path_policy_events.jsonl"
POST_MOVE_ADL_LOG = WORKING_DIR / "post_move_adl_events.jsonl"
DIFFERENCE_MEMORY_LOG = WORKING_DIR / "adl_difference_memory.jsonl"

# The utility is deliberately interpretable. Current-game empirical evidence is
# allowed to override these priors after observed transitions accumulate.
ADL_FUSION_WEIGHTS = {
    "legality": 0.15,
    "predicted_progress": 0.18,
    "predicted_frame_change": 0.10,
    "information_gain": 0.13,
    "novelty": 0.10,
    "causal_consistency": 0.12,
    "world_model_consistency": 0.10,
    "action_efficiency": 0.07,
    "loop_avoidance": 0.05,
}
assert abs(sum(ADL_FUSION_WEIGHTS.values()) - 1.0) < 1e-9

@dataclass
class ADLDifferenceSignature:
    step: int
    action: str
    state_changed: str = "uncertain"
    score_delta: Optional[float] = None
    level_delta: Optional[float] = None
    prediction_match: str = "uncertain"
    information_gain: float = 0.0
    progress_value: float = 0.0
    loop_signal: bool = False
    novel_transition: str = "uncertain"
    lesson: str = ""
    next_bias: str = "neutral"
    evidence_strength: float = 0.0

    def validate(self) -> None:
        if self.step < 0:
            raise ValueError("ADL step must be non-negative")
        if not -1.0 <= float(self.progress_value) <= 1.0:
            raise ValueError("progress_value must be in [-1, 1]")
        if not 0.0 <= float(self.information_gain) <= 1.0:
            raise ValueError("information_gain must be in [0, 1]")
        if not 0.0 <= float(self.evidence_strength) <= 1.0:
            raise ValueError("evidence_strength must be in [0, 1]")
        if self.next_bias not in {"exploit", "explore", "neutral"}:
            raise ValueError("next_bias must be exploit/explore/neutral")


class CurrentGameDifferenceMemory:
    """Current-game-only ADL ledger. Never reused across games."""
    def __init__(self, game_id: str):
        self.game_id = str(game_id)
        self.records: List[ADLDifferenceSignature] = []
        self.action_values: Dict[str, List[float]] = {}
        self.loop_counts: Dict[str, int] = {}
        self.prediction_hits = 0
        self.prediction_total = 0

    def add(self, record: ADLDifferenceSignature) -> None:
        record.validate()
        self.records.append(record)
        self.action_values.setdefault(record.action, []).append(float(record.progress_value))
        if record.loop_signal:
            self.loop_counts[record.action] = self.loop_counts.get(record.action, 0) + 1
        if record.prediction_match in {"yes", "partial", "no"}:
            self.prediction_total += 1
            if record.prediction_match == "yes":
                self.prediction_hits += 1

    def empirical_action_value(self, action: str) -> float:
        values = self.action_values.get(action, ())
        return sum(values) / len(values) if values else 0.0

    def prediction_accuracy(self) -> float:
        return self.prediction_hits / self.prediction_total if self.prediction_total else 0.0

    def compact_context(self, limit: int = 8) -> str:
        recent = self.records[-max(1, int(limit)):]
        payload = {
            "schema": ADL_SCHEMA,
            "game_id": self.game_id,
            "records_seen": len(self.records),
            "prediction_accuracy": round(self.prediction_accuracy(), 4),
            "loop_counts": self.loop_counts,
            "recent_differences": [asdict(r) for r in recent],
        }
        return json.dumps(payload, sort_keys=True, separators=(",", ":"))


def adl_fusion_utility(metrics: Dict[str, float]) -> float:
    """Weighted utility for comparing two hypothetical candidates only.

    This function does NOT step or simulate the environment. It only combines
    estimates made from the current observation/current-game evidence.
    """
    total = 0.0
    for key, weight in ADL_FUSION_WEIGHTS.items():
        value = float(metrics.get(key, 0.0))
        if not 0.0 <= value <= 1.0:
            raise ValueError(f"{key} must be in [0,1], got {value}")
        total += weight * value
    return total


DUCKWADL_ADL_CONTRACT = r"""
DUCKWADL ADL DIFFERENCEFUSION CONTRACT — REQUIRED FOR EVERY REAL MOVE

This is ONE real ARC-AGI-3 environment trajectory. Never fork, clone, replay,
reset for speculation, or execute a second environment to compare candidates.
Candidate comparison is INTERNAL reasoning over the SAME current observation.
Use only evidence from THIS CURRENT GAME and THIS CURRENT RUN.

============================================================
0 — VISIBLE CURRENT-GAME DIFFERENCE STATE
============================================================
Before every real action, visibly maintain:
World model:
Goal model:
Action model:
Recent findings:
Open questions:
Difference memory:
Prediction calibration:
Known no-ops/loops:
Cross-level notes from this SAME game only:

The Difference memory is a compact set of transformations, not just facts:
  state/context + action/change -> observed delta -> value/confidence.
Never import a lesson from another game.

============================================================
1 — EXACTLY TWO CANDIDATES FROM THE SAME CURRENT STATE
============================================================
Candidate A = EXPLOIT
- shortest legal action supported by confirmed current-game evidence
- maximize expected progress
- avoid known loops, no-ops, regressions, deaths, and redundant probes

Candidate B = EXPLORE
- legal action with highest expected information gain
- target unresolved mechanics, objects, transitions, or controls
- avoid exhausted probes and known loops

For EACH candidate estimate numbers in [0,1]:
LEGality
PREDICTED_PROGRESS
PREDICTED_FRAME_CHANGE
INFORMATION_GAIN
NOVELTY
CAUSAL_CONSISTENCY
WORLD_MODEL_CONSISTENCY
ACTION_EFFICIENCY
LOOP_AVOIDANCE

Use weighted DifferenceFusion:
legality=.15
predicted_progress=.18
predicted_frame_change=.10
information_gain=.13
novelty=.10
causal_consistency=.12
world_model_consistency=.10
action_efficiency=.07
loop_avoidance=.05

Current-game empirical evidence overrides generic assumptions.

Before the action visibly emit:
[DUCKWADL][PLAN]
DUAL_PATH_DECISION:
STEP=<integer>
STATE_SIGNATURE=<compact description/hash-like signature>
A_ACTION=<candidate A>
A_PREDICTION=<expected state/progress delta>
A_COMPONENTS=<metric=value,...>
A_FUSION=<0..1>
B_ACTION=<candidate B>
B_PREDICTION=<expected state/progress delta>
B_COMPONENTS=<metric=value,...>
B_FUSION=<0..1>
PREDICTED_DIFFERENCE=<why A and B are behaviorally different>
SELECT=<A or B>
WHY=<evidence-grounded reason>

Then visibly emit:
[DUCKWADL][ACTION] STEP=<integer> ACTION=<exact committed action/data>

Issue exactly ONE real environment action.

============================================================
2 — IMMEDIATE POST-MOVE AI DIFFERENCE LEARNING
============================================================
As soon as the real tool result returns and BEFORE the next plan, compare:
- pre-state vs post-state
- predicted delta vs actual delta
- selected candidate vs rejected candidate at the hypothesis level only
- current observation vs analogous states encountered earlier THIS GAME

Visibly emit:
[DUCKWADL][RESULT]
POST_MOVE_ADL:
STEP=<same integer>
ACTION=<actual committed action>
STATE_CHANGED=<yes/no/uncertain>
STATE_DELTA=<compact semantic description>
SCORE_DELTA=<number if observed, otherwise unknown>
LEVEL_DELTA=<number if observed, otherwise unknown>
PREDICTION_MATCH=<yes/partial/no/uncertain>
INFORMATION_GAIN=<0..1>
PROGRESS_VALUE=<-1..1>
LOOP_SIGNAL=<yes/no>
NOVEL_TRANSITION=<yes/no/uncertain>
DIFFERENCE_SIGNATURE=<context + action -> observed delta>
LESSON=<compact causal/current-game-only lesson>
EVIDENCE_STRENGTH=<0..1>
NEXT_BIAS=<exploit/explore/neutral>

The POST_MOVE_ADL record MUST alter the next candidate estimates when relevant.
Examples:
- positive progress: increase value/confidence of that local transformation
- unchanged state: downweight repeating the same action in equivalent states
- regression/death: strongly downweight that local pattern
- useful surprise: increase exploration value for related unresolved actions
- prediction miss: reduce confidence in the assumption that generated it
- repeated agreement: increase evidence strength, but never to certainty from one hit

============================================================
3 — ADL DIFFERENCE LEARNING OBJECTIVE
============================================================
Do not merely imitate a successful-looking action. Learn:
  DELTA_INPUT/STATE + DELTA_ACTION/STRATEGY -> DELTA_OUTCOME

Choose the smallest current-game-supported change expected to improve the next
outcome. Distinguish correlation from causal evidence. Prefer repeated local
observations over one-off guesses.

============================================================
4 — STRICT NO-PRIOR BOUNDARY
============================================================
Allowed:
- generic model capability
- current-game observations/actions/rewards/transitions
- current-game visible world model and difference records
- current-run runtime/tool information

Forbidden:
- prior games
- prior submissions
- stored winning routes
- external solution memory
- replay libraries
- hidden labels
- game source-code introspection
- cross-game ADL memory
- speculative second-environment steps

At a new game, begin with EMPTY game-specific Difference Memory.
""".strip()


class DuckWADLToolAgent(ToolAgent):
    """Duck ToolAgent with mandatory pre-action DifferenceFusion and post-move ADL."""
    def __init__(self, game_id: str = "unknown", **kwargs):
        super().__init__(**kwargs)
        self._adl_memory = CurrentGameDifferenceMemory(game_id)
        if DUCKWADL_ADL_CONTRACT not in self._system_prompt:
            self._system_prompt = self._system_prompt.rstrip() + "\n\n" + DUCKWADL_ADL_CONTRACT


def _duckwadl_adl_analyzer_factory(game, index):
    model = (
        os.environ.get("INFERENCE_ANALYZER_MODEL")
        or os.environ.get("LOCAL_ANALYZER_MODEL_ID")
        or "vrfai/Qwen3.6-27B-FP8"
    )
    base_url = (
        os.environ.get("LOCAL_ANALYZER_BASE_URL")
        or os.environ.get("OPENAI_BASE_URL")
        or "http://127.0.0.1:1234/v1"
    )
    game_id = getattr(game, "game_id", None) or getattr(game, "id", None) or f"game-{index}"
    return DuckWADLToolAgent(
        game_id=str(game_id),
        model=model,
        timeout=bm.solver.analyzer_timeout,
        save_request_logs=bm.solver.save_request_logs,
        base_url=base_url,
        provider="vllm",
    )


bm.solver.analyzer_factory = _duckwadl_adl_analyzer_factory

print("DUCKWADL ADL DIFFERENCEFUSION ACTIVE", flush=True)
print(f"ADL_SCHEMA={ADL_SCHEMA}", flush=True)
print("BEFORE EVERY MOVE: exactly 2 internal candidates from one current state", flush=True)
print("FUSION: structured weighted candidate comparison", flush=True)
print("AFTER EVERY MOVE: predicted-vs-actual semantic delta + confidence update", flush=True)
print("ENVIRONMENT PASSES PER GAME: 1", flush=True)
print("CROSS-GAME DIFFERENCE MEMORY: DISABLED", flush=True)


## 7.1 ADL integration self-test

Validates DifferenceFusion normalization, difference-record constraints, and current-game memory behavior before any environment action is executed.


In [ ]:
# === STATIC ADL SELF-TEST: NO ENVIRONMENT ACTIONS ===
_test_memory = CurrentGameDifferenceMemory("self-test")
_test_record = ADLDifferenceSignature(
    step=0,
    action="ACTION_TEST",
    state_changed="yes",
    prediction_match="partial",
    information_gain=0.7,
    progress_value=0.4,
    loop_signal=False,
    novel_transition="yes",
    lesson="test-only schema validation",
    next_bias="exploit",
    evidence_strength=0.6,
)
_test_memory.add(_test_record)
assert _test_memory.empirical_action_value("ACTION_TEST") == 0.4
assert _test_memory.prediction_total == 1
assert 0.0 <= adl_fusion_utility({k: 0.5 for k in ADL_FUSION_WEIGHTS}) <= 1.0
assert len(_test_memory.records) == 1
# Do not expose or persist self-test memory to the real game factory.
del _test_memory, _test_record
print("ADL SELF-TEST PASSED: schema, fusion weights, and isolated memory")


## 8. Run exactly one real environment trajectory per game

Competition and local modes share the same policy. Competition mode discovers games
from the official gateway. Local mode uses the mounted public `environment_files`.


In [ ]:
# === ONE-ENVIRONMENT COMPETITION/LOCAL EXECUTION ===
import re
from urllib.request import urlopen

EXPECTED_LOCAL_GAME_COUNT = 25


def _game_key(value):
    text = str(value or "").strip()
    match = re.match(r"([A-Za-z0-9]+)", text)
    if not match:
        raise ValueError(f"Cannot derive game key from {value!r}")
    return match.group(1)


def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed no games.")
    if len(set(game_ids)) != len(game_ids):
        raise RuntimeError("Competition Arcade exposed duplicate game IDs.")
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _offline_games():
    import arc_agi
    import taaf.game_api

    root = Path(
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"
    )
    if not root.is_dir():
        raise FileNotFoundError(f"Local environment root missing: {root}")

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if len(game_ids) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Expected {EXPECTED_LOCAL_GAME_COUNT} local games; found {len(game_ids)}"
        )
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _wait_for_gateway(base_url, timeout_s=900):
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Competition gateway did not become ready: {last_error}")


def _run_score(run):
    return float(getattr(run, "final_score", None) or 0.0)


def _run_levels(run):
    return int(getattr(run, "levels_completed", 0) or 0)


def _run_actions(run):
    return len(getattr(run, "history", ()) or ())


def _won(run):
    state = getattr(run, "state", "")
    return str(getattr(state, "name", state)).lower().endswith("won")


if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    game_apis = _competition_games()
    print(
        f"OFFICIAL COMPETITION MODE: {len(game_apis)} games, "
        "one environment trajectory per game",
        flush=True,
    )
else:
    game_apis = _offline_games()
    print(
        f"LOCAL MODE: {len(game_apis)} games, one environment trajectory per game",
        flush=True,
    )

RUN_GAME_COUNT = len(game_apis)
if RUN_GAME_COUNT < 1:
    raise RuntimeError("No ARC-AGI-3 games discovered.")

# One pass only. No hidden-game restart and no environment best-of-two.
bm.games = game_apis
bm.n_passes = 1
bm.game_weights = None
bm.solver.concurrency = TARGET_CONCURRENCY

# Runtime budget: reserve setup/teardown time, then divide the remaining time
# across the one legal environment pass.
NOTEBOOK_RUNTIME_TARGET_SECONDS = 8 * 60 * 60 + 30 * 60
NON_ENVIRONMENT_RESERVE_SECONDS = 70 * 60
available_environment_seconds = (
    NOTEBOOK_RUNTIME_TARGET_SECONDS - NON_ENVIRONMENT_RESERVE_SECONDS
)
runtime_safe_per_game = max(
    45.0,
    available_environment_seconds * TARGET_CONCURRENCY / RUN_GAME_COUNT,
)
RUN_PER_GAME_SECONDS = min(
    1500.0,
    runtime_safe_per_game,
    _original_game_budget if _original_game_budget > 0 else runtime_safe_per_game,
)
bm.solver.max_runtime_s_per_game = RUN_PER_GAME_SECONDS

run_manifest = {
    "schema": "adl.arc3.dual-path.clean.v1",
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": RUN_GAME_COUNT,
    "environment_passes_per_game": 1,
    "internal_candidate_plans_per_decision": 2,
    "concurrency": TARGET_CONCURRENCY,
    "per_game_seconds": RUN_PER_GAME_SECONDS,
    "strict_no_prior": True,
    "second_environment_pass": False,
}

(WORKING_DIR / "dual_path_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(
    "DUAL-PATH RUN START "
    f"games={RUN_GAME_COUNT} concurrency={TARGET_CONCURRENCY} "
    f"per_game_seconds={RUN_PER_GAME_SECONDS:.1f}",
    flush=True,
)

try:
    await bm.run(
        soft_end_time=None,
        runtime_environment=target,
        minimal_diagnostics=bool(TRUE_SUBMISSION),
    )
finally:
    # Keep the source bundle's own teardown behavior.
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print("Running teardown:", command, flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )

if len(getattr(bm, "game_runs", []) or []) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Expected {RUN_GAME_COUNT} completed game runs; "
        f"found {len(getattr(bm, 'game_runs', []) or [])}"
    )

for run in bm.game_runs:
    print(
        "DUAL-PATH SCORE "
        f"game={_game_key(run.game_id)} "
        f"score={_run_score(run):.6f} "
        f"levels={_run_levels(run)} "
        f"actions={_run_actions(run)}",
        flush=True,
    )


## 9. Write and validate `submission.parquet`


In [ ]:
# === VALIDATED COMPETITION ARTIFACT ===
import pandas as pd

rows = [
    {
        "row_id": f"{run.game_id}_0",
        "game_id": str(run.game_id),
        "end_of_game": _won(run),
        "score": _run_score(run),
    }
    for run in bm.game_runs
]

if len(rows) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Submission requires {RUN_GAME_COUNT} rows; found {len(rows)}"
    )

submission = pd.DataFrame(
    rows,
    columns=["row_id", "game_id", "end_of_game", "score"],
)

if submission["game_id"].astype(str).nunique() != RUN_GAME_COUNT:
    raise RuntimeError("Submission contains duplicate game IDs.")
if submission["score"].isna().any():
    raise RuntimeError("Submission contains missing scores.")

SUBMISSION_PATH = WORKING_DIR / "submission.parquet"
submission.to_parquet(SUBMISSION_PATH, index=False)

check = pd.read_parquet(SUBMISSION_PATH)
if list(check.columns) != ["row_id", "game_id", "end_of_game", "score"]:
    raise RuntimeError(f"Invalid submission columns: {list(check.columns)}")
if len(check) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Written submission row count mismatch: {len(check)} != {RUN_GAME_COUNT}"
    )

print(
    "SUBMISSION READY "
    f"path={SUBMISSION_PATH} rows={len(check)} "
    f"score_sum={float(check['score'].sum()):.6f}",
    flush=True,
)


## 10. Final ADL run summary


In [ ]:
# === FINAL CLEAN ADL SUMMARY ===
import json
import re

runs = list(bm.game_runs)
scores = [_run_score(run) for run in runs]
levels = [_run_levels(run) for run in runs]
actions = [_run_actions(run) for run in runs]

summary = {
    "schema": "adl.arc3.dual-path.clean.v1",
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": len(runs),
    "mean_score": (sum(scores) / len(scores) if scores else 0.0),
    "positive_score_games": sum(score > 0 for score in scores),
    "total_levels_completed": sum(levels),
    "total_actions": sum(actions),
    "concurrency": TARGET_CONCURRENCY,
    "environment_passes_per_game": 1,
    "internal_candidate_plans_per_decision": 2,
    "second_environment_pass": False,
    "strict_no_prior": True,
    "submission_path": str(SUBMISSION_PATH),
}

summary_path = WORKING_DIR / "dual_path_adl_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("=" * 72)
print("ARC-AGI-3 DUAL-PATH ADL — CLEAN RUN")
print(f"competition_rerun={TRUE_SUBMISSION}")
print("2 internal plans -> 1 selected action -> 1 environment trajectory")
print("SECOND ENVIRONMENT PASS: DISABLED")
print(
    f"games={summary['games']} "
    f"mean_score={summary['mean_score']:.6f} "
    f"positive_games={summary['positive_score_games']} "
    f"levels={summary['total_levels_completed']} "
    f"actions={summary['total_actions']}"
)
print(f"submission={SUBMISSION_PATH}")
print(f"summary={summary_path}")
print("=" * 72)


## 11. Structured DuckWADL ADL audit

Parses the visible per-move DifferenceFusion and `POST_MOVE_ADL` traces into a current-run difference dataset, measures coverage against committed actions, and reports prediction calibration, progress, information gain, loops, and novel transitions. The generated difference-memory file is an audit artifact and is **not** loaded across games.


In [ ]:
# === DUCKWADL STRUCTURED ADL EXTRACTION / AUDIT ===
# Converts visible PLAN/POST_MOVE_ADL traces from THIS run into a structured
# difference-learning dataset, while verifying that ADL covered the real actions.

import json
import re
from pathlib import Path

_TEXT_EXTS = {".log", ".txt", ".json", ".jsonl", ".md"}
_SKIP_NAMES = {
    "dual_path_adl_summary.json",
    "post_move_adl_audit.json",
    "duckwadl_adl_audit.json",
    "adl_difference_memory.jsonl",
}


def _adl_text_files(root: Path):
    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in _TEXT_EXTS:
            continue
        if path.name in _SKIP_NAMES:
            continue
        yield path


def _safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _extract_blocks(text: str, marker: str):
    # Capture marker payload until next blank boundary/major tagged event.
    pattern = re.compile(
        re.escape(marker) + r"\s*\n(?P<body>.*?)(?=\n\[(?:DUCKWADL|DIFFERENCEFUSION)\]|\n(?:DUAL_PATH_DECISION:|POST_MOVE_ADL:)|\Z)",
        re.S,
    )
    return [m.group("body") for m in pattern.finditer(text)]


def _parse_key_values(body: str):
    out = {}
    for line in body.splitlines():
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip().upper()
        value = value.strip()
        if key:
            out[key] = value
    return out


plan_records = []
post_records = []
files_scanned = []
for path in _adl_text_files(WORKING_DIR):
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    plan_bodies = _extract_blocks(text, "DUAL_PATH_DECISION:")
    post_bodies = _extract_blocks(text, "POST_MOVE_ADL:")
    if plan_bodies or post_bodies:
        files_scanned.append(str(path))
    plan_records.extend(_parse_key_values(body) for body in plan_bodies)
    post_records.extend(_parse_key_values(body) for body in post_bodies)

# De-duplicate repeated logger copies conservatively by key fields.
def _dedupe(records, keys):
    seen = set()
    unique = []
    for rec in records:
        sig = tuple(rec.get(k, "") for k in keys)
        if sig in seen:
            continue
        seen.add(sig)
        unique.append(rec)
    return unique

plan_records = _dedupe(plan_records, ("STEP", "A_ACTION", "B_ACTION", "SELECT"))
post_records = _dedupe(post_records, ("STEP", "ACTION", "DIFFERENCE_SIGNATURE", "LESSON"))

# Emit an explicit current-run ADL dataset. It is an audit artifact only; it is
# never loaded into another game by this notebook.
with DIFFERENCE_MEMORY_LOG.open("w", encoding="utf-8") as f:
    for rec in post_records:
        row = {
            "schema": ADL_SCHEMA,
            "step": int(rec["STEP"]) if rec.get("STEP", "").isdigit() else rec.get("STEP"),
            "action": rec.get("ACTION"),
            "state_changed": rec.get("STATE_CHANGED"),
            "state_delta": rec.get("STATE_DELTA"),
            "score_delta": _safe_float(rec.get("SCORE_DELTA")),
            "level_delta": _safe_float(rec.get("LEVEL_DELTA")),
            "prediction_match": rec.get("PREDICTION_MATCH"),
            "information_gain": _safe_float(rec.get("INFORMATION_GAIN")),
            "progress_value": _safe_float(rec.get("PROGRESS_VALUE")),
            "loop_signal": rec.get("LOOP_SIGNAL"),
            "novel_transition": rec.get("NOVEL_TRANSITION"),
            "difference_signature": rec.get("DIFFERENCE_SIGNATURE"),
            "lesson": rec.get("LESSON"),
            "evidence_strength": _safe_float(rec.get("EVIDENCE_STRENGTH")),
            "next_bias": rec.get("NEXT_BIAS"),
        }
        f.write(json.dumps(row, sort_keys=True) + "\n")

runs = list(getattr(bm, "game_runs", []) or [])
total_actions = sum(len(getattr(run, "history", ()) or ()) for run in runs)

matches = [r.get("PREDICTION_MATCH", "").lower() for r in post_records]
known_predictions = [m for m in matches if m in {"yes", "partial", "no"}]
exact_prediction_accuracy = (
    sum(m == "yes" for m in known_predictions) / len(known_predictions)
    if known_predictions else None
)
soft_prediction_accuracy = (
    sum(1.0 if m == "yes" else 0.5 if m == "partial" else 0.0 for m in known_predictions)
    / len(known_predictions)
    if known_predictions else None
)
progress_values = [
    value for value in (_safe_float(r.get("PROGRESS_VALUE")) for r in post_records)
    if value is not None
]
info_values = [
    value for value in (_safe_float(r.get("INFORMATION_GAIN")) for r in post_records)
    if value is not None
]

post_move_coverage = len(post_records) / total_actions if total_actions else 0.0
plan_coverage = len(plan_records) / total_actions if total_actions else 0.0

audit = {
    "schema": ADL_SCHEMA,
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": len(runs),
    "total_actions": total_actions,
    "dual_path_decisions": len(plan_records),
    "post_move_adl_updates": len(post_records),
    "plan_coverage": plan_coverage,
    "post_move_coverage": post_move_coverage,
    "prediction_exact_accuracy": exact_prediction_accuracy,
    "prediction_soft_accuracy": soft_prediction_accuracy,
    "mean_progress_value": sum(progress_values) / len(progress_values) if progress_values else None,
    "mean_information_gain": sum(info_values) / len(info_values) if info_values else None,
    "loop_signals": sum(str(r.get("LOOP_SIGNAL", "")).lower() == "yes" for r in post_records),
    "novel_transitions": sum(str(r.get("NOVEL_TRANSITION", "")).lower() == "yes" for r in post_records),
    "difference_memory_rows": len(post_records),
    "difference_memory_path": str(DIFFERENCE_MEMORY_LOG),
    "files_with_adl_trace": sorted(set(files_scanned)),
    "single_environment_pass": True,
    "cross_game_memory": False,
    "required_policy": "pre-action DifferenceFusion + POST_MOVE_ADL after every committed real move",
}

audit_path = WORKING_DIR / "duckwadl_adl_audit.json"
audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("=" * 72)
print("DUCKWADL ADL AUDIT")
print(json.dumps(audit, indent=2, sort_keys=True))
print(f"difference_memory={DIFFERENCE_MEMORY_LOG}")
print(f"audit={audit_path}")
print("=" * 72)
